# SARIMAX nacional — inadimplência e desastres climáticos

Esta etapa complementa Granger e DLM com regressão temporal de erros SARIMA. Usamos a série nacional **já auditada** e versionada em `outputs/tables/segunda_etapa/series_nacionais_agregadas.csv`. O período completo é 2013–2024; o pré-pandemia até janeiro de 2020 é descritivo. A unidade é Brasil×mês.

O SARIMA univariado é a referência. Para os SARIMAX, cada modelo contém uma única exposição climática e um lag 0, 1 ou 3, selecionado por BIC. A análise é observacional e as inferências condicionadas à seleção são exploratórias.

## 1. Dados, transformações e critério de parcimônia

O indicador nacional corresponde a `100 × ∑ carteira inadimplente / ∑ carteira ativa`. A variável resposta é `100 Δlog(I)` e cada exposição usa `Δlog(1+D)`. As diferenças são externas ao SARIMAX; os modelos usam `d=0` e `D=0`. A ausência de diferenciação sazonal é uma escolha parcimoniosa a ser verificada pelos diagnósticos, não uma conclusão automática dos testes.

O código usa sete estruturas ARMA simples e, no máximo, uma ordem AR sazonal. Resíduos, convergência, suporte e comparação temporal complementam os critérios de informação.

In [ ]:
from pathlib import Path
import runpy

raiz = Path.cwd()
if not (raiz / 'src' / 'sarimax.py').exists():
    raiz = raiz.parent
assert (raiz / 'src' / 'sarimax.py').exists(), 'Execute na raiz do repositório ou em notebooks/'
runpy.run_path(str(raiz / 'src' / 'sarimax.py'), run_name='__main__')

## 2. Resultados e diagnósticos

A seleção usa apenas treino até dezembro de 2022. A avaliação principal usa janeiro de 2023 a dezembro de 2024 e origem móvel de um passo à frente com parâmetros fixos. Para `lag=0`, a previsão é **condicional**, pois utiliza o desastre do mesmo mês já observado. Não deve ser vendida como previsão operacional.

Os modelos pré-pandemia usam amostra menor; as inferências são exploratórias. Os valores-p usuais e a correção BH não incorporam a incerteza da seleção do lag/especificação.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

saida = raiz / 'outputs' / 'tables' / 'sarimax'
for nome in ['modelos','coeficientes','diagnosticos','metricas','estacionariedade']:
    print('\n', nome)
    display(pd.read_csv(saida / f'{nome}.csv').round(4))
print('Relatório:', raiz / 'outputs' / 'reports' / 'relatorio_sarimax.html')

## 3. Interpretação e limitações

O coeficiente de `Δlog(1+D)` está na escala de **100×Δlog da inadimplência**: não é variação em pontos percentuais da taxa, tampouco efeito causal. Lag isolado não representa efeito acumulado. A comparação SARIMA×SARIMAX permite avaliar ganho condicional de ajuste e previsão, não identificação causal.

As tabelas nacionais agregadas não permitem recuperar exatamente os grupos COBRADE sem registros brutos. O total climático é validado como códigos 12/13/14; tipologias de baixa frequência são avaliadas quanto ao suporte. Ver `docs/sarimax_protocolo.md`.